# Analysis — When Does Kubernetes Become Worth It?

This notebook aggregates all experiment data and produces the tables and figures used in the thesis.

**Logical chain:**
```
SQ1 (What breaks on the VM?)
  → SQ2 (Does Kubernetes fix it?)
    → SQ3 (What does Kubernetes cost?)
      → SQ4 (Where do the lines cross?)
```

**Data expected in:**
- `data/raw/exp1-vm-degradation/L*/run*/dagster_runs.csv`
- `data/raw/exp2-kubernetes-isolation/part-a/L*/run*/dagster_runs.csv`
- `data/raw/exp2-kubernetes-isolation/part-a/L*/run*/pod_timing.csv`
- `data/raw/exp2-kubernetes-isolation/part-b-blast-radius/*/run*/blast_radius.csv`

**Outputs written to:** `data/processed/` and `results/`

## 0 · Setup

In [1]:
import os, glob, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_RAW    = os.path.join(REPO_ROOT, 'data', 'raw')
DATA_PROC   = os.path.join(REPO_ROOT, 'data', 'processed')
RESULTS_DIR = os.path.join(REPO_ROOT, 'results')

os.makedirs(DATA_PROC,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':       150,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        11,
})

LEVELS       = [1, 2, 3, 5, 7, 10]
LEVEL_LABELS = ['L1\n(1)', 'L2\n(2)', 'L3\n(3)', 'L4\n(5)', 'L5\n(7)', 'L6\n(10)']

print(f'Repo root : {REPO_ROOT}')
print(f'Data (raw): {DATA_RAW}')
print(f'Results   : {RESULTS_DIR}')

ModuleNotFoundError: No module named 'pandas'

## 1 · Load Data

In [2]:
d
e
f
 
l
o
a
d
_
d
a
g
s
t
e
r
_
r
u
n
s
(
e
x
p
e
r
i
m
e
n
t
_
d
i
r
:
 
s
t
r
)
 
-
>
 
p
d
.
D
a
t
a
F
r
a
m
e
:

 
 
 
 
"
"
"
L
o
a
d
 
a
l
l
 
d
a
g
s
t
e
r
_
r
u
n
s
.
c
s
v
 
f
i
l
e
s
 
f
r
o
m
 
a
n
 
e
x
p
e
r
i
m
e
n
t
 
d
i
r
e
c
t
o
r
y
 
t
r
e
e
.
"
"
"

 
 
 
 
f
r
a
m
e
s
 
=
 
[
]

 
 
 
 
f
o
r
 
m
e
t
a
_
p
a
t
h
 
i
n
 
g
l
o
b
.
g
l
o
b
(
o
s
.
p
a
t
h
.
j
o
i
n
(
e
x
p
e
r
i
m
e
n
t
_
d
i
r
,
 
'
*
*
'
,
 
'
m
e
t
a
d
a
t
a
.
j
s
o
n
'
)
,
 
r
e
c
u
r
s
i
v
e
=
T
r
u
e
)
:

 
 
 
 
 
 
 
 
r
u
n
_
d
i
r
 
=
 
o
s
.
p
a
t
h
.
d
i
r
n
a
m
e
(
m
e
t
a
_
p
a
t
h
)

 
 
 
 
 
 
 
 
r
u
n
s
_
c
s
v
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
r
u
n
_
d
i
r
,
 
'
d
a
g
s
t
e
r
_
r
u
n
s
.
c
s
v
'
)

 
 
 
 
 
 
 
 
i
f
 
n
o
t
 
o
s
.
p
a
t
h
.
e
x
i
s
t
s
(
r
u
n
s
_
c
s
v
)
:

 
 
 
 
 
 
 
 
 
 
 
 
c
o
n
t
i
n
u
e

 
 
 
 
 
 
 
 
w
i
t
h
 
o
p
e
n
(
m
e
t
a
_
p
a
t
h
)
 
a
s
 
f
:

 
 
 
 
 
 
 
 
 
 
 
 
m
e
t
a
 
=
 
j
s
o
n
.
l
o
a
d
(
f
)

 
 
 
 
 
 
 
 
d
f
 
=
 
p
d
.
r
e
a
d
_
c
s
v
(
r
u
n
s
_
c
s
v
)

 
 
 
 
 
 
 
 
d
f
[
'
l
e
v
e
l
'
]
 
 
 
 
 
 
 
=
 
m
e
t
a
[
'
c
o
n
c
u
r
r
e
n
t
_
j
o
b
s
'
]

 
 
 
 
 
 
 
 
d
f
[
'
r
e
p
e
t
i
t
i
o
n
'
]
 
 
=
 
m
e
t
a
.
g
e
t
(
'
r
e
p
'
,
 
m
e
t
a
.
g
e
t
(
'
r
e
p
e
t
i
t
i
o
n
'
,
 
0
)
)

 
 
 
 
 
 
 
 
d
f
[
'
e
n
v
i
r
o
n
m
e
n
t
'
]
 
=
 
m
e
t
a
.
g
e
t
(
'
e
n
v
'
,
 
m
e
t
a
.
g
e
t
(
'
e
n
v
i
r
o
n
m
e
n
t
'
,
 
'
u
n
k
n
o
w
n
'
)
)

 
 
 
 
 
 
 
 
#
 
C
o
m
p
u
t
e
 
d
u
r
a
t
i
o
n
_
s
e
c
o
n
d
s
 
f
r
o
m
 
s
t
a
r
t
_
t
i
m
e
 
a
n
d
 
e
n
d
_
t
i
m
e

 
 
 
 
 
 
 
 
i
f
 
'
s
t
a
r
t
_
t
i
m
e
'
 
i
n
 
d
f
.
c
o
l
u
m
n
s
 
a
n
d
 
'
e
n
d
_
t
i
m
e
'
 
i
n
 
d
f
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
d
f
[
'
s
t
a
r
t
_
t
i
m
e
'
]
 
=
 
p
d
.
t
o
_
d
a
t
e
t
i
m
e
(
d
f
[
'
s
t
a
r
t
_
t
i
m
e
'
]
,
 
u
t
c
=
T
r
u
e
,
 
e
r
r
o
r
s
=
'
c
o
e
r
c
e
'
)

 
 
 
 
 
 
 
 
 
 
 
 
d
f
[
'
e
n
d
_
t
i
m
e
'
]
 
 
 
=
 
p
d
.
t
o
_
d
a
t
e
t
i
m
e
(
d
f
[
'
e
n
d
_
t
i
m
e
'
]
,
 
 
 
u
t
c
=
T
r
u
e
,
 
e
r
r
o
r
s
=
'
c
o
e
r
c
e
'
)

 
 
 
 
 
 
 
 
 
 
 
 
d
f
[
'
d
u
r
a
t
i
o
n
_
s
e
c
o
n
d
s
'
]
 
=
 
(
d
f
[
'
e
n
d
_
t
i
m
e
'
]
 
-
 
d
f
[
'
s
t
a
r
t
_
t
i
m
e
'
]
)
.
d
t
.
t
o
t
a
l
_
s
e
c
o
n
d
s
(
)

 
 
 
 
 
 
 
 
f
r
a
m
e
s
.
a
p
p
e
n
d
(
d
f
)

 
 
 
 
r
e
t
u
r
n
 
p
d
.
c
o
n
c
a
t
(
f
r
a
m
e
s
,
 
i
g
n
o
r
e
_
i
n
d
e
x
=
T
r
u
e
)
 
i
f
 
f
r
a
m
e
s
 
e
l
s
e
 
p
d
.
D
a
t
a
F
r
a
m
e
(
)



d
e
f
 
l
o
a
d
_
p
o
d
_
t
i
m
i
n
g
(
e
x
p
e
r
i
m
e
n
t
_
d
i
r
:
 
s
t
r
)
 
-
>
 
p
d
.
D
a
t
a
F
r
a
m
e
:

 
 
 
 
"
"
"
L
o
a
d
 
a
l
l
 
p
o
d
_
t
i
m
i
n
g
.
c
s
v
 
f
i
l
e
s
 
a
n
d
 
c
o
m
p
u
t
e
 
d
e
r
i
v
e
d
 
l
a
t
e
n
c
y
 
c
o
l
u
m
n
s
.
"
"
"

 
 
 
 
f
r
a
m
e
s
 
=
 
[
]

 
 
 
 
f
o
r
 
m
e
t
a
_
p
a
t
h
 
i
n
 
g
l
o
b
.
g
l
o
b
(
o
s
.
p
a
t
h
.
j
o
i
n
(
e
x
p
e
r
i
m
e
n
t
_
d
i
r
,
 
'
*
*
'
,
 
'
m
e
t
a
d
a
t
a
.
j
s
o
n
'
)
,
 
r
e
c
u
r
s
i
v
e
=
T
r
u
e
)
:

 
 
 
 
 
 
 
 
r
u
n
_
d
i
r
 
 
 
=
 
o
s
.
p
a
t
h
.
d
i
r
n
a
m
e
(
m
e
t
a
_
p
a
t
h
)

 
 
 
 
 
 
 
 
t
i
m
i
n
g
_
c
s
v
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
r
u
n
_
d
i
r
,
 
'
p
o
d
_
t
i
m
i
n
g
.
c
s
v
'
)

 
 
 
 
 
 
 
 
i
f
 
n
o
t
 
o
s
.
p
a
t
h
.
e
x
i
s
t
s
(
t
i
m
i
n
g
_
c
s
v
)
:

 
 
 
 
 
 
 
 
 
 
 
 
c
o
n
t
i
n
u
e

 
 
 
 
 
 
 
 
w
i
t
h
 
o
p
e
n
(
m
e
t
a
_
p
a
t
h
)
 
a
s
 
f
:

 
 
 
 
 
 
 
 
 
 
 
 
m
e
t
a
 
=
 
j
s
o
n
.
l
o
a
d
(
f
)

 
 
 
 
 
 
 
 
d
f
 
=
 
p
d
.
r
e
a
d
_
c
s
v
(
t
i
m
i
n
g
_
c
s
v
)

 
 
 
 
 
 
 
 
d
f
[
'
l
e
v
e
l
'
]
 
 
 
 
 
 
=
 
m
e
t
a
[
'
c
o
n
c
u
r
r
e
n
t
_
j
o
b
s
'
]

 
 
 
 
 
 
 
 
d
f
[
'
r
e
p
e
t
i
t
i
o
n
'
]
 
=
 
m
e
t
a
.
g
e
t
(
'
r
e
p
'
,
 
m
e
t
a
.
g
e
t
(
'
r
e
p
e
t
i
t
i
o
n
'
,
 
0
)
)

 
 
 
 
 
 
 
 
#
 
P
a
r
s
e
 
t
i
m
e
s
t
a
m
p
s

 
 
 
 
 
 
 
 
f
o
r
 
c
o
l
 
i
n
 
[
'
s
u
b
m
i
t
t
e
d
_
t
s
'
,
 
'
s
c
h
e
d
u
l
e
d
_
t
s
'
,
 
'
r
u
n
n
i
n
g
_
t
s
'
,
 
'
j
o
b
_
s
t
a
r
t
_
t
s
'
]
:

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
c
o
l
 
i
n
 
d
f
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
d
f
[
c
o
l
]
 
=
 
p
d
.
t
o
_
d
a
t
e
t
i
m
e
(
d
f
[
c
o
l
]
,
 
u
t
c
=
T
r
u
e
,
 
e
r
r
o
r
s
=
'
c
o
e
r
c
e
'
)

 
 
 
 
 
 
 
 
#
 
s
c
h
e
d
u
l
i
n
g
_
l
a
t
e
n
c
y
_
s
:
 
s
u
b
m
i
t
t
e
d
 
→
 
p
o
d
 
s
c
h
e
d
u
l
e
d
 
b
y
 
K
8
s

 
 
 
 
 
 
 
 
i
f
 
'
s
u
b
m
i
t
t
e
d
_
t
s
'
 
i
n
 
d
f
.
c
o
l
u
m
n
s
 
a
n
d
 
'
s
c
h
e
d
u
l
e
d
_
t
s
'
 
i
n
 
d
f
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
d
f
[
'
s
c
h
e
d
u
l
i
n
g
_
l
a
t
e
n
c
y
_
s
'
]
 
=
 
(
d
f
[
'
s
c
h
e
d
u
l
e
d
_
t
s
'
]
 
-
 
d
f
[
'
s
u
b
m
i
t
t
e
d
_
t
s
'
]
)
.
d
t
.
t
o
t
a
l
_
s
e
c
o
n
d
s
(
)

 
 
 
 
 
 
 
 
#
 
s
t
a
r
t
u
p
_
l
a
t
e
n
c
y
_
s
:
 
s
c
h
e
d
u
l
e
d
 
→
 
c
o
n
t
a
i
n
e
r
s
 
r
e
a
d
y
 
(
r
u
n
n
i
n
g
_
t
s
)

 
 
 
 
 
 
 
 
i
f
 
'
s
c
h
e
d
u
l
e
d
_
t
s
'
 
i
n
 
d
f
.
c
o
l
u
m
n
s
 
a
n
d
 
'
r
u
n
n
i
n
g
_
t
s
'
 
i
n
 
d
f
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
d
f
[
'
s
t
a
r
t
u
p
_
l
a
t
e
n
c
y
_
s
'
]
 
=
 
(
d
f
[
'
r
u
n
n
i
n
g
_
t
s
'
]
 
-
 
d
f
[
'
s
c
h
e
d
u
l
e
d
_
t
s
'
]
)
.
d
t
.
t
o
t
a
l
_
s
e
c
o
n
d
s
(
)

 
 
 
 
 
 
 
 
#
 
t
o
t
a
l
_
o
v
e
r
h
e
a
d
_
s
:
 
s
u
b
m
i
t
t
e
d
 
→
 
j
o
b
 
a
c
t
u
a
l
l
y
 
e
x
e
c
u
t
i
n
g

 
 
 
 
 
 
 
 
i
f
 
'
s
u
b
m
i
t
t
e
d
_
t
s
'
 
i
n
 
d
f
.
c
o
l
u
m
n
s
 
a
n
d
 
'
j
o
b
_
s
t
a
r
t
_
t
s
'
 
i
n
 
d
f
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
d
f
[
'
t
o
t
a
l
_
o
v
e
r
h
e
a
d
_
s
'
]
 
=
 
(
d
f
[
'
j
o
b
_
s
t
a
r
t
_
t
s
'
]
 
-
 
d
f
[
'
s
u
b
m
i
t
t
e
d
_
t
s
'
]
)
.
d
t
.
t
o
t
a
l
_
s
e
c
o
n
d
s
(
)

 
 
 
 
 
 
 
 
f
r
a
m
e
s
.
a
p
p
e
n
d
(
d
f
)

 
 
 
 
r
e
t
u
r
n
 
p
d
.
c
o
n
c
a
t
(
f
r
a
m
e
s
,
 
i
g
n
o
r
e
_
i
n
d
e
x
=
T
r
u
e
)
 
i
f
 
f
r
a
m
e
s
 
e
l
s
e
 
p
d
.
D
a
t
a
F
r
a
m
e
(
)



d
e
f
 
l
o
a
d
_
b
l
a
s
t
_
r
a
d
i
u
s
(
e
n
v
i
r
o
n
m
e
n
t
:
 
s
t
r
)
 
-
>
 
p
d
.
D
a
t
a
F
r
a
m
e
:

 
 
 
 
"
"
"
L
o
a
d
 
b
l
a
s
t
_
r
a
d
i
u
s
.
c
s
v
 
f
i
l
e
s
 
f
o
r
 
v
m
 
o
r
 
k
8
s
.
"
"
"

 
 
 
 
b
l
a
s
t
_
d
i
r
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
D
A
T
A
_
R
A
W
,
 
'
e
x
p
2
-
k
u
b
e
r
n
e
t
e
s
-
i
s
o
l
a
t
i
o
n
'
,
 
'
p
a
r
t
-
b
-
b
l
a
s
t
-
r
a
d
i
u
s
'
,
 
e
n
v
i
r
o
n
m
e
n
t
)

 
 
 
 
f
r
a
m
e
s
 
=
 
[
]

 
 
 
 
f
o
r
 
c
s
v
_
p
a
t
h
 
i
n
 
g
l
o
b
.
g
l
o
b
(
o
s
.
p
a
t
h
.
j
o
i
n
(
b
l
a
s
t
_
d
i
r
,
 
'
*
*
'
,
 
'
b
l
a
s
t
_
r
a
d
i
u
s
.
c
s
v
'
)
,
 
r
e
c
u
r
s
i
v
e
=
T
r
u
e
)
:

 
 
 
 
 
 
 
 
f
r
a
m
e
s
.
a
p
p
e
n
d
(
p
d
.
r
e
a
d
_
c
s
v
(
c
s
v
_
p
a
t
h
)
)

 
 
 
 
r
e
t
u
r
n
 
p
d
.
c
o
n
c
a
t
(
f
r
a
m
e
s
,
 
i
g
n
o
r
e
_
i
n
d
e
x
=
T
r
u
e
)
 
i
f
 
f
r
a
m
e
s
 
e
l
s
e
 
p
d
.
D
a
t
a
F
r
a
m
e
(
)



#
 
─
─
 
L
o
a
d
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

v
m
_
d
a
t
a
 
 
 
 
=
 
l
o
a
d
_
d
a
g
s
t
e
r
_
r
u
n
s
(
o
s
.
p
a
t
h
.
j
o
i
n
(
D
A
T
A
_
R
A
W
,
 
'
e
x
p
1
-
v
m
-
d
e
g
r
a
d
a
t
i
o
n
'
)
)

k
8
s
_
d
a
t
a
 
 
 
=
 
l
o
a
d
_
d
a
g
s
t
e
r
_
r
u
n
s
(
o
s
.
p
a
t
h
.
j
o
i
n
(
D
A
T
A
_
R
A
W
,
 
'
e
x
p
2
-
k
u
b
e
r
n
e
t
e
s
-
i
s
o
l
a
t
i
o
n
'
,
 
'
p
a
r
t
-
a
'
)
)

t
i
m
i
n
g
_
d
a
t
a
 
=
 
l
o
a
d
_
p
o
d
_
t
i
m
i
n
g
(
o
s
.
p
a
t
h
.
j
o
i
n
(
D
A
T
A
_
R
A
W
,
 
'
e
x
p
2
-
k
u
b
e
r
n
e
t
e
s
-
i
s
o
l
a
t
i
o
n
'
,
 
'
p
a
r
t
-
a
'
)
)

b
l
a
s
t
_
v
m
 
 
 
=
 
l
o
a
d
_
b
l
a
s
t
_
r
a
d
i
u
s
(
'
v
m
'
)

b
l
a
s
t
_
k
8
s
 
 
=
 
l
o
a
d
_
b
l
a
s
t
_
r
a
d
i
u
s
(
'
k
8
s
'
)


p
r
i
n
t
(
f
'
V
M
 
r
u
n
s
 
 
 
:
 
{
l
e
n
(
v
m
_
d
a
t
a
)
}
 
 
r
o
w
s
'
)

p
r
i
n
t
(
f
'
K
8
s
 
r
u
n
s
 
 
:
 
{
l
e
n
(
k
8
s
_
d
a
t
a
)
}
 
r
o
w
s
'
)

p
r
i
n
t
(
f
'
P
o
d
 
t
i
m
i
n
g
:
 
{
l
e
n
(
t
i
m
i
n
g
_
d
a
t
a
)
}
 
r
o
w
s
'
)

p
r
i
n
t
(
f
'
B
l
a
s
t
 
(
V
M
)
:
 
{
l
e
n
(
b
l
a
s
t
_
v
m
)
}
 
r
o
w
s
'
)

p
r
i
n
t
(
f
'
B
l
a
s
t
 
(
K
8
s
)
:
 
{
l
e
n
(
b
l
a
s
t
_
k
8
s
)
}
 
r
o
w
s
'
)

SyntaxError: unterminated string literal (detected at line 65) (4177373334.py, line 65)

## 2 · SQ1 — VM Degradation Profile

> *At what concurrency level does a single-VM deployment fail, and how do job success rate, execution time variance, and resource utilisation degrade toward that threshold?*

In [3]:
def summary_table(df: pd.DataFrame) -> pd.DataFrame:
    """Compute per-level summary statistics."""
    if df.empty:
        return pd.DataFrame()
    return (
        df.groupby('level')
        .agg(
            total_runs    = ('run_id', 'count'),
            success_count = ('status', lambda x: (x == 'SUCCESS').sum()),
            success_rate  = ('status', lambda x: (x == 'SUCCESS').mean() * 100),
            mean_s        = ('duration_seconds', 'mean'),
            std_s         = ('duration_seconds', 'std'),
            min_s         = ('duration_seconds', 'min'),
            max_s         = ('duration_seconds', 'max'),
        )
        .reindex(LEVELS)
        .round(2)
    )


vm_summary = summary_table(vm_data)
vm_summary.to_csv(os.path.join(DATA_PROC, 'vm_summary.csv'))

display(vm_summary.style
    .format({'success_rate': '{:.1f}%', 'mean_s': '{:.2f}s', 'std_s': '{:.2f}s'})
    .background_gradient(subset=['success_rate'], cmap='RdYlGn', vmin=0, vmax=100)
    .set_caption('Table 1 — VM Degradation Summary'))

NameError: name 'vm_data' is not defined

In [4]:
if not vm_summary.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    # Success rate curve
    ax1.plot(vm_summary.index, vm_summary['success_rate'],
             'o-', color='#d62728', linewidth=2.5, markersize=8, label='VM success rate')
    ax1.axhline(95, color='gray', linestyle='--', linewidth=1.2, label='95% threshold')
    ax1.fill_between(vm_summary.index, vm_summary['success_rate'], 95,
                     where=vm_summary['success_rate'] < 95, alpha=0.12, color='red', label='Degradation zone')
    ax1.set_xticks(LEVELS); ax1.set_xticklabels(LEVEL_LABELS)
    ax1.set_xlabel('Concurrent Jobs'); ax1.set_ylabel('Success Rate (%)')
    ax1.set_title('SQ1 — VM Reliability Degradation'); ax1.set_ylim(0, 105)
    ax1.legend(fontsize=9)

    # Execution time ± std dev
    ax2.errorbar(vm_summary.index, vm_summary['mean_s'], yerr=vm_summary['std_s'],
                 fmt='o-', color='#d62728', linewidth=2.5, capsize=5, markersize=8, label='VM mean ± std')
    ax2.set_xticks(LEVELS); ax2.set_xticklabels(LEVEL_LABELS)
    ax2.set_xlabel('Concurrent Jobs'); ax2.set_ylabel('Execution Time (s)')
    ax2.set_title('SQ1 — VM Execution Time vs Concurrency')
    ax2.legend(fontsize=9)

    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, 'exp1-vm-degradation-curve.png')
    plt.savefig(out, bbox_inches='tight'); plt.show()
    print(f'Saved → {out}')
else:
    print('⚠ No VM data yet — run Experiment 1 first.')

NameError: name 'vm_summary' is not defined

## 3 · SQ2 — Kubernetes Isolation & Blast Radius

> *To what degree does Kubernetes pod isolation contain failures and reduce execution time variance compared to the single-VM deployment under equivalent concurrent workload levels?*

In [5]:
k8s_summary = summary_table(k8s_data)
k8s_summary.to_csv(os.path.join(DATA_PROC, 'k8s_summary.csv'))

display(k8s_summary.style
    .format({'success_rate': '{:.1f}%', 'mean_s': '{:.2f}s', 'std_s': '{:.2f}s'})
    .background_gradient(subset=['success_rate'], cmap='RdYlGn', vmin=0, vmax=100)
    .set_caption('Table 2 — K8s Isolation Summary'))

NameError: name 'k8s_data' is not defined

In [6]:
if not vm_summary.empty and not k8s_summary.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    for ax, metric, ylabel, title in [
        (ax1, 'success_rate', 'Success Rate (%)', 'SQ2 — Reliability: VM vs K8s'),
        (ax2, 'mean_s',       'Execution Time (s)', 'SQ2 — Performance: VM vs K8s'),
    ]:
        ax.plot(vm_summary.index,  vm_summary[metric],  'o-', color='#d62728',
                linewidth=2.5, markersize=8, label='VM')
        ax.plot(k8s_summary.index, k8s_summary[metric], 's-', color='#1f77b4',
                linewidth=2.5, markersize=8, label='K8s')
        ax.set_xticks(LEVELS); ax.set_xticklabels(LEVEL_LABELS)
        ax.set_xlabel('Concurrent Jobs'); ax.set_ylabel(ylabel)
        ax.set_title(title); ax.legend(fontsize=9)
        if metric == 'success_rate':
            ax.axhline(95, color='gray', linestyle='--', linewidth=1.2, alpha=0.7)
            ax.set_ylim(0, 105)

    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, 'vm-vs-k8s-comparison.png')
    plt.savefig(out, bbox_inches='tight'); plt.show()
    print(f'Saved → {out}')
else:
    print('⚠ Need both VM (Exp1) and K8s (Exp2A) data.')

NameError: name 'vm_summary' is not defined

In [7]:
# Blast radius comparison (Exp2B)
if not blast_vm.empty and not blast_k8s.empty:
    blast_vm ['env'] = 'VM'
    blast_k8s['env'] = 'K8s'
    blast = pd.concat([blast_vm, blast_k8s], ignore_index=True)

    summary_blast = blast.groupby('environment').agg(
        mean_affected = ('affected_jobs', 'mean'),
        std_affected  = ('affected_jobs', 'std'),
    ).round(2)

    display(summary_blast.style.set_caption('Table 3 — Blast Radius at L4 (mean affected jobs)'))

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(summary_blast.index, summary_blast['mean_affected'],
           color=['#d62728', '#1f77b4'], yerr=summary_blast['std_affected'], capsize=5)
    ax.set_ylabel('Mean Jobs Affected'); ax.set_title('SQ2 — Blast Radius at L4')
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, 'blast-radius.png')
    plt.savefig(out, bbox_inches='tight'); plt.show()
    print(f'Saved → {out}')
else:
    print('⚠ No blast radius data yet — run Experiment 2B first.')

NameError: name 'blast_vm' is not defined

## 4 · SQ3 — Kubernetes Scheduling Overhead

> *What measurable scheduling latency and execution overhead does Kubernetes introduce compared to the VM process executor at each concurrency level?*

In [8]:
if not timing_data.empty:
    overhead = (
        timing_data.groupby('level')
        .agg(
            mean_sched_s   = ('scheduling_latency_s', 'mean'),
            std_sched_s    = ('scheduling_latency_s', 'std'),
            mean_startup_s = ('startup_latency_s',    'mean'),
            std_startup_s  = ('startup_latency_s',    'std'),
            mean_total_s   = ('total_overhead_s',     'mean'),
            std_total_s    = ('total_overhead_s',     'std'),
        )
        .reindex(LEVELS)
        .round(3)
    )
    overhead.to_csv(os.path.join(DATA_PROC, 'overhead_summary.csv'))

    display(overhead.style
        .format('{:.3f}s')
        .background_gradient(subset=['mean_total_s'], cmap='YlOrRd')
        .set_caption('Table 4 — K8s Scheduling Overhead per Level'))

    fig, ax = plt.subplots(figsize=(9, 4.5))
    w = 0.35
    x = np.arange(len(overhead))
    ax.bar(x - w/2, overhead['mean_sched_s'],   w, yerr=overhead['std_sched_s'],
           label='Scheduling latency', color='#ff7f0e', capsize=4)
    ax.bar(x + w/2, overhead['mean_startup_s'], w, yerr=overhead['std_startup_s'],
           label='Container startup',  color='#2ca02c', capsize=4)
    ax.set_xticks(x); ax.set_xticklabels(LEVEL_LABELS)
    ax.set_xlabel('Concurrent Jobs'); ax.set_ylabel('Overhead (s)')
    ax.set_title('SQ3 — K8s Scheduling Overhead by Level')
    ax.legend(fontsize=9)
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, 'k8s-scheduling-overhead.png')
    plt.savefig(out, bbox_inches='tight'); plt.show()
    print(f'Saved → {out}')
else:
    print('⚠ No pod timing data yet — run Experiment 2A first.')

NameError: name 'timing_data' is not defined

## 5 · SQ4 — The Crossover Point

> *At what concurrency level does the Kubernetes overhead become smaller than the VM's contention-induced degradation — the formally defined crossover point?*

In [9]:
if not vm_summary.empty and not k8s_summary.empty:
    crossover = pd.DataFrame({
        'vm_success_%':  vm_summary['success_rate'],
        'k8s_success_%': k8s_summary['success_rate'],
        'vm_mean_s':     vm_summary['mean_s'],
        'k8s_mean_s':    k8s_summary['mean_s'],
    }, index=LEVELS)
    crossover['delta_s']      = (crossover['vm_mean_s'] - crossover['k8s_mean_s']).round(2)
    crossover['faster']       = crossover['delta_s'].apply(lambda d: 'K8s' if d > 0 else ('VM' if d < 0 else 'Tie'))
    crossover['k8s_reliable'] = crossover['k8s_success_%'] >= 95
    crossover['k8s_faster']   = crossover['delta_s'] > 0
    crossover['net_beneficial'] = crossover['k8s_reliable'] & crossover['k8s_faster']
    crossover.to_csv(os.path.join(DATA_PROC, 'crossover_table.csv'))

    display(crossover.style
        .format({'vm_success_%': '{:.1f}%', 'k8s_success_%': '{:.1f}%',
                 'vm_mean_s': '{:.2f}s',    'k8s_mean_s': '{:.2f}s', 'delta_s': '{:+.2f}s'})
        .applymap(lambda v: 'background-color: #c6efce' if v is True else
                            ('background-color: #ffc7ce' if v is False else ''),
                  subset=['net_beneficial'])
        .set_caption('Table 5 — VM vs K8s Crossover Table (SQ4)'))
else:
    print('⚠ Need both VM and K8s summary data.')

NameError: name 'vm_summary' is not defined

In [10]:
if not vm_summary.empty and not k8s_summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(vm_summary.index,  vm_summary['mean_s'],  'o-', color='#d62728',
            linewidth=2.5, markersize=9, label='VM (Process Executor)', zorder=3)
    ax.fill_between(vm_summary.index, vm_summary['mean_s'] - vm_summary['std_s'],
                    vm_summary['mean_s'] + vm_summary['std_s'], alpha=0.15, color='#d62728')

    ax.plot(k8s_summary.index, k8s_summary['mean_s'], 's-', color='#1f77b4',
            linewidth=2.5, markersize=9, label='K8s (K8sRunLauncher)', zorder=3)
    ax.fill_between(k8s_summary.index, k8s_summary['mean_s'] - k8s_summary['std_s'],
                    k8s_summary['mean_s'] + k8s_summary['std_s'], alpha=0.15, color='#1f77b4')

    # Mark crossover
    vm_t   = vm_summary['mean_s'].values
    k8s_t  = k8s_summary['mean_s'].values
    levels = np.array(LEVELS)
    for i in range(1, len(levels)):
        if vm_t[i] > k8s_t[i] and vm_t[i-1] <= k8s_t[i-1]:
            cx = (levels[i-1] + levels[i]) / 2
            ax.axvline(cx, color='green', linestyle='--', linewidth=2,
                       label=f'Crossover ≈ {cx:.0f} jobs', zorder=4)
            break

    ax.set_xticks(LEVELS); ax.set_xticklabels(LEVEL_LABELS)
    ax.set_xlabel('Concurrent Jobs', fontsize=12)
    ax.set_ylabel('Mean Execution Time (s)', fontsize=12)
    ax.set_title('SQ4 — The Crossover Point: When K8s Becomes Worth It', fontsize=13)
    ax.legend(fontsize=10)
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, 'crossover-plot.png')
    plt.savefig(out, bbox_inches='tight'); plt.show()
    print(f'Saved → {out}')
else:
    print('⚠ Need both VM and K8s data to compute crossover.')

NameError: name 'vm_summary' is not defined

## 6 · Statistical Significance

Mann-Whitney U test at each concurrency level (non-parametric, appropriate for small samples).

In [11]:
if not vm_data.empty and not k8s_data.empty:
    rows = []
    for level in LEVELS:
        vm_t  = vm_data [vm_data ['level'] == level]['duration_seconds'].dropna()
        k8s_t = k8s_data[k8s_data['level'] == level]['duration_seconds'].dropna()
        if len(vm_t) >= 2 and len(k8s_t) >= 2:
            U, p = stats.mannwhitneyu(vm_t, k8s_t, alternative='two-sided')
            rows.append({'level': level, 'vm_n': len(vm_t), 'k8s_n': len(k8s_t),
                         'U': round(U, 1), 'p_value': round(p, 4),
                         'significant (p<0.05)': 'Yes' if p < 0.05 else 'No'})

    p_table = pd.DataFrame(rows)
    p_table.to_csv(os.path.join(DATA_PROC, 'p_values.csv'), index=False)
    display(p_table.style
        .applymap(lambda v: 'color: green; font-weight: bold' if v == 'Yes' else
                            'color: gray' if v == 'No' else '',
                  subset=['significant (p<0.05)'])
        .set_caption('Table 6 — Statistical Significance (Mann-Whitney U)'))
else:
    print('⚠ Need both VM and K8s data for statistical tests.')

NameError: name 'vm_data' is not defined

## 7 · Export Summary

All outputs saved to `data/processed/` and `results/`.

In [12]:
from pathlib import Path

print('─── data/processed/ ────────────────────────────────')
for f in sorted(Path(DATA_PROC).glob('*.csv')):
    print(f'  {f.name}')

print()
print('─── results/ ────────────────────────────────────────')
for f in sorted(Path(RESULTS_DIR).glob('*.png')):
    print(f'  {f.name}')

print()
print('Run `make copy-figures` to copy PNGs into docs/figures/ for LaTeX.')

─── data/processed/ ────────────────────────────────


NameError: name 'DATA_PROC' is not defined